# Fundamentals 09 - Graph API

Objetivo: separar `Graph` de `System`. Un graph coordina nodos y estado. Puede usar agentes como nodos con `agent.as_node(...)` o `lab.agent_node(...)`, pero no reemplaza al system ni al agent.

Modelo mental:

```text
Graph = state + nodes + edges + run
```


In [ ]:
from __future__ import annotations

import importlib.util
from typing import Any, TypedDict

import agentic_systems as lab

PRETTY = False
HAS_LANGGRAPH = importlib.util.find_spec("langgraph") is not None

runtime = lab.runtime(provider="python-direct", model="local-python", region="local")

## Problema default de fundamentals

Todos los notebooks de `tutorials/` usan este mismo problema para comparar la API sin cambiar de caso:

```text
Empieza con 10, suma 20, resta 9, multiplica por 4 y divide entre 2.
Dime:
- procedimiento
- resultado final
```


In [ ]:
USER_PROMPT = """Empieza con 10, suma 20, resta 9, multiplica por 4 y divide entre 2.
Dime:
- procedimiento
- resultado final""".strip()

REQUESTED_OUTPUTS = ["procedimiento", "resultado_final"]

lab.show({
    "prompt_usuario": USER_PROMPT,
    "salidas_solicitadas": REQUESTED_OUTPUTS,
}, title="Problema default")

## 1) Definir estado del graph

El graph mueve un `state`: cada nodo recibe un dict y devuelve un update parcial. Eso permite auditar qué dato produjo cada paso.

In [ ]:
class ArithmeticGraphState(TypedDict, total=False):
    prompt: str
    current_value: float
    procedure: list[str]
    result: lab.RunResult
    final_answer: dict[str, Any]

initial_state: ArithmeticGraphState = {
    "prompt": USER_PROMPT,
    "current_value": 10,
    "procedure": [],
}

lab.show(initial_state, title="Initial graph state")

## 2) Crear agentes deterministas para nodos

Estos agentes usan `python-direct`: cada nodo ejecuta una tool concreta. Esto muestra `agent.as_node(...)` como puente natural entre Agent y Graph.

In [ ]:
@lab.tool
def sumar(a: float, b: float) -> dict:
    return {"operation": "sumar", "result": a + b, "explanation": f"{a} + {b} = {a + b}"}


@lab.tool
def restar(a: float, b: float) -> dict:
    return {"operation": "restar", "result": a - b, "explanation": f"{a} - {b} = {a - b}"}


@lab.tool
def multiplicar(a: float, b: float) -> dict:
    return {"operation": "multiplicar", "result": a * b, "explanation": f"{a} * {b} = {a * b}"}


@lab.tool
def dividir(a: float, b: float) -> dict:
    if b == 0:
        raise ValueError("No se puede dividir entre cero.")
    value = a / b
    return {"operation": "dividir", "result": value, "explanation": f"{a} / {b} = {value}"}


def make_tool_agent(name: str, tool: Any) -> lab.Agent:
    return lab.agent(
        name=name,
        instructions=f"Ejecuta la tool {tool.name} cuando el state lo solicite.",
        tools=[tool],
        engine="python-direct",
        runtime=runtime,
        contract=lab.AgentContract(must_call=[tool.name], completion="when_required_tools_satisfied"),
        policy=lab.RunPolicy(max_tool_calls=1, max_turns=1),
    )

agents = {
    "sumar": make_tool_agent("sumar_node_agent", sumar),
    "restar": make_tool_agent("restar_node_agent", restar),
    "multiplicar": make_tool_agent("multiplicar_node_agent", multiplicar),
    "dividir": make_tool_agent("dividir_node_agent", dividir),
}

lab.show({name: agent.info() for name, agent in agents.items()}, title="Graph node agents")

## 3) Convertir agentes a nodos

Cada mapper lee `state`, llama al agente y devuelve un update. El graph no calcula negocio oculto: solo coordina pasos.

In [ ]:
def tool_input(tool_name: str, amount: float):
    return lambda state: {"tool": tool_name, "input": {"a": state["current_value"], "b": amount}}


def node_output(result: lab.RunResult, state: dict[str, Any]) -> dict[str, Any]:
    data = result.data
    return {
        "current_value": data["result"],
        "procedure": [*state.get("procedure", []), data["explanation"]],
        "result": result,
    }


nodes = {
    "sumar": lab.agent_node(agents["sumar"], input=tool_input("sumar", 20), output=node_output, result_key=None, mode="eval"),
    "restar": lab.agent_node(agents["restar"], input=tool_input("restar", 9), output=node_output, result_key=None, mode="eval"),
    "multiplicar": lab.agent_node(agents["multiplicar"], input=tool_input("multiplicar", 4), output=node_output, result_key=None, mode="eval"),
    "dividir": lab.agent_node(agents["dividir"], input=tool_input("dividir", 2), output=node_output, result_key=None, mode="eval"),
}

lab.show({"nodes": list(nodes)}, title="Graph nodes")

## 4) Ejecutar como graph nativo si LangGraph está instalado

`lab.graph(...)` usa LangGraph como backend opcional. Si no está instalado, el notebook muestra el pipeline de nodos con el mismo contrato `state -> update`.

In [ ]:
edges = [
    ("START", "sumar"),
    ("sumar", "restar"),
    ("restar", "multiplicar"),
    ("multiplicar", "dividir"),
    ("dividir", "END"),
]

state = dict(initial_state)

if HAS_LANGGRAPH:
    graph = lab.graph(
        state=ArithmeticGraphState,
        nodes=nodes,
        edges=edges,
        engine="langgraph",
        name="fundamentals_arithmetic_graph",
    )
    state = graph.run(state)
else:
    for tool_name in ["sumar", "restar", "multiplicar", "dividir"]:
        state.update(nodes[tool_name](state))

final_answer = {
    "procedimiento": state["procedure"],
    "resultado_final": state["current_value"],
}
state["final_answer"] = final_answer

lab.show({
    "backend": "langgraph" if HAS_LANGGRAPH else "local state pipeline",
    "final_answer": state["final_answer"],
    "state_keys": list(state.keys()),
}, title="Graph run")

## 5) Human result y lineage del graph

El graph produce state; `RunResult` y `LineageMemory` hacen visible qué pasó y qué respuesta se entregó.

In [ ]:
graph_result = lab.RunResult(
    text="El graph resolvio el problema default.",
    data=state["final_answer"],
    final=state["final_answer"],
    engine="python-direct",
    model="local-python",
    mode="graph",
    meta={"framework": "langgraph" if HAS_LANGGRAPH else "local-state-graph", "state_keys": list(state)},
)

lineage = graph_result.lineage(
    name="fundamentals.graph.arithmetic",
    question=USER_PROMPT,
    goal="Explicar el recorrido state -> nodes -> final_answer.",
)

lab.human_result(
    graph_result,
    title="Human result - Graph API",
    pretty=PRETTY,
    show_lineage=True,
    lineage=lineage,
)

## Lo importante

- `Graph` coordina estado, nodos y edges.
- `agent.as_node(...)` y `lab.agent_node(...)` son el puente natural entre Agent y Graph.
- Un graph puede ser single-agent, multi-agent o pipeline de nodos deterministas.
- `framework="langgraph"` pertenece a integraciones; `Graph API` explica el contrato conceptual.

## Coverage API de este notebook

Esta tabla deja explicito que parte de Agentic Systems queda materializada aqui.

In [ ]:
api_coverage = [
    {"api": "lab.agent_node", "description": "Convierte un Agent en nodo state -> update."},
    {"api": "agent.as_node", "description": "Expresa el mismo puente desde el propio agente."},
    {"api": "lab.graph", "description": "Compone nodes y edges cuando LangGraph esta disponible."},
    {"api": "graph.run(state)", "description": "Ejecuta la orquestacion sobre un state declarativo."},
    {"api": "RunResult + LineageMemory", "description": "Vuelve auditable la salida final del graph."},
]

lab.show({"notebook": "09_graph_api.ipynb", "api_coverage": api_coverage})